In [1]:
%reload_ext autoreload
%autoreload 1

# M5 Forecast Linear Regression Per Item-Store

## Imports

In [2]:
import time
import os
import warnings
import numpy as np
import multiprocessing
import pandas as pd
from pathlib import Path

id_col = "id"
time_col = "date"
id_cols = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']

PATH_INPUT = Path("data/m5/datasets")
N_JOBS = multiprocessing.cpu_count()

HORIZON = 28  # How many days into the future to predict
VAL_DAYS = 0  # How many days of data to hold-out for validatioin (set to 0 for live)

if not PATH_INPUT.exists():
    raise FileNotFoundError("Please download the M5 dataset first by executing `src/generate_data.py`")

warnings.simplefilter("ignore")
os.environ["NIXTLA_ID_AS_COL"] = '1'
start_time = time.time()

# Helper Functions

In [3]:
from typing import List
from tqdm import tqdm
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from src.process import *

# Load Inputs

In [4]:
cal = load_calendar(PATH_INPUT)
prices = load_prices(PATH_INPUT)

df = pd.read_parquet("data/train.snap.parquet")

2025-03-30 21:31:46.430 | DEBUG    | src.process:load_calendar:30 - Begin Loading Calendar Data...
2025-03-30 21:31:46.493 | DEBUG    | src.process:load_prices:44 - Begin Loading Price Data...


## Try Fitting Linear Regression for Each SKU

In [5]:
# Feature Engineering: Extract time-based features
df["day"] = df["date"].dt.day
df["month"] = df["date"].dt.month
df["weekday"] = df["date"].dt.weekday
df["weekofyear"] = df["date"].dt.isocalendar().week

# Create lag features (e.g., past sales as features)
for lag in [1, 7, 14, 28]:
    df[f"y_lag_{lag}"] = df.groupby("id")["y"].shift(lag)

# Drop NaN values after lag creation
df = df.dropna()

# Select features
features = ["day", "month", "weekday", "weekofyear", "sell_price", "y_lag_1", "y_lag_7", "y_lag_14", "y_lag_28"]

# Forecast Horizon
forecast_horizon = 28  # Predict next 28 days

# Store predictions
forecast_results = []

# Train a separate model for each SKU
for i, sku_id in tqdm(enumerate(df["id"].unique())):
    
    if i % 1_000 == 0:
        logger.info(f"""Training model {i} of {df["id"].nunique()} for SKU: {sku_id}""")
    
    # Filter data for this SKU
    df_sku = df[df["id"] == sku_id].copy()
    
    # Split into training and testing sets (last 28 days for evaluation)
    train_data = df_sku.iloc[:-forecast_horizon]
    test_data = df_sku.iloc[-forecast_horizon:]
    
    X_train, y_train = train_data[features], train_data["y"]
    X_test, y_test = test_data[features], test_data["y"]
    
    # Train a Linear Regression model
    model = LinearRegression()
    model.fit(X_train, y_train)
    
    # Store last known row to generate future predictions
    last_row = df_sku.iloc[-1][features].copy()
    
    # Generate future dates
    future_dates = pd.date_range(start=df_sku["date"].max() + pd.Timedelta(days=1), periods=forecast_horizon, freq="D")
    
    # Create empty dataframe to store future predictions
    future_df = pd.DataFrame({"date": future_dates})
    
    # Copy over known features that do not change
    future_df["day"] = future_df["date"].dt.day
    future_df["month"] = future_df["date"].dt.month
    future_df["weekday"] = future_df["date"].dt.weekday
    future_df["weekofyear"] = future_df["date"].dt.isocalendar().week
    future_df["sell_price"] = last_row["sell_price"]  # Assuming price remains the same
    
    # Recursive forecasting
    predictions = []
    for i in range(forecast_horizon):
        # Create new row with updated lag values
        new_row = future_df.iloc[i].copy()
        
        # Assign lag values from previous predictions
        if i > 0:
            new_row["y_lag_1"] = predictions[-1]  # Previous prediction as lag_1
        else:
            new_row["y_lag_1"] = last_row["y_lag_1"]
        
        new_row["y_lag_7"] = last_row["y_lag_7"] if i < 7 else predictions[-7]
        new_row["y_lag_14"] = last_row["y_lag_14"] if i < 14 else predictions[-14]
        new_row["y_lag_28"] = last_row["y_lag_28"] if i < 28 else predictions[-28]

        # Convert new_row to dataframe for model prediction
        X_future = new_row[features].values.reshape(1, -1)
        pred = model.predict(X_future)[0]
        predictions.append(pred)

    # Store results
    future_df["forecast"] = predictions
    future_df["id"] = sku_id
    forecast_results.append(future_df)



2025-03-30 21:31:59.077 | INFO     | __main__:<module>:27 - Training model 0 of 30490 for SKU: FOODS_1_001_CA_1_evaluation
2025-03-30 21:34:10.451 | INFO     | __main__:<module>:27 - Training model 1000 of 30490 for SKU: FOODS_1_103_CA_1_evaluation
2025-03-30 21:36:24.415 | INFO     | __main__:<module>:27 - Training model 2000 of 30490 for SKU: FOODS_1_204_CA_1_evaluation
2025-03-30 21:38:19.820 | INFO     | __main__:<module>:27 - Training model 3000 of 30490 for SKU: FOODS_2_085_CA_1_evaluation
2025-03-30 21:40:13.892 | INFO     | __main__:<module>:27 - Training model 4000 of 30490 for SKU: FOODS_2_186_CA_1_evaluation
2025-03-30 21:42:12.370 | INFO     | __main__:<module>:27 - Training model 5000 of 30490 for SKU: FOODS_2_286_CA_1_evaluation
2025-03-30 21:45:45.926 | INFO     | __main__:<module>:27 - Training model 6000 of 30490 for SKU: FOODS_2_386_CA_1_evaluation
2025-03-30 21:48:10.585 | INFO     | __main__:<module>:27 - Training model 7000 of 30490 for SKU: FOODS_3_088_CA_1_evalua

In [6]:
# Combine all forecasts
forecast_df = pd.concat(forecast_results, ignore_index=True)
logger.info(f"""Finished forecasting: {forecast_df["id"].nunique():,d}""")

2025-03-30 22:33:27.427 | INFO     | __main__:<module>:3 - Finished forecasting: 30,490


In [7]:
forecast_df = forecast_df.rename(columns={"forecast": "forecast_lr"})

In [8]:
# forecast_df = pd.read_parquet("data/lr_predictions.snap.parquet")
# forecast_df

In [9]:
def make_submission(preds: pd.DataFrame,
                    h: int) -> pd.DataFrame:
    wide = preds.pivot_table(index='id',
                             columns='date',
                             observed=True)
    wide.columns = [f'F{i+1}' for i in range(h)]
    wide.columns.name = None
    wide.index.name = 'id'
    return wide



In [10]:
from datasetsforecast.m5 import M5Evaluation

dfids = get_dfids()
submission = make_submission(forecast_df[["id", "date", "forecast_lr"]], HORIZON)
# submission = submission.join(dfids.set_index("id"))

df_score = M5Evaluation.evaluate("data", submission.join(dfids.set_index("id")))
model_score = df_score["wrmsse"].mean()
df_score

,wrmsse
Total,0.775114
Level1,0.522048
Level2,0.599686
Level3,0.703741
Level4,0.577883
Level5,0.690135
Level6,0.677562
Level7,0.773639
Level8,0.766767
Level9,0.838091


In [11]:
submission.to_parquet("data/submission_linear_reg.snap.parquet")

In [12]:
print(f"""Linear Regression wrmsse: {model_score:.4f}""")
print(f"""Notebook finished in {(time.time() - start_time) / 60:.2f}m""")

Linear Regression wrmsse: 0.7751
Notebook finished in 62.08m
